In [2]:
import pandas as pd 
!pip install mysql-connector-python

import mysql.connector 
from sqlalchemy import create_engine
from getpass import getpass
!pip install PyMySQL

In [36]:
def rentals_month(engine, month, year):
    cursor = engine.cursor()
    month_number = pd.to_datetime(month, format='%B').month
    
    query = f"""
        SELECT * 
        FROM rental 
        WHERE YEAR(rental_date) = {year} 
        AND MONTH(rental_date) = {month_number};
    """
    
    cursor.execute(query)
    results = cursor.fetchall()
    
    # Create a DataFrame from the results
    df = pd.DataFrame(results)
    df = pd.DataFrame(results, columns = [header[0] for header in cursor.description])
    
    return df

In [105]:
#Test the function
engine = mysql.connector.connect(user="root", password=password, host="localhost", database="sakila")
month = "May"
year = "2005"

df_rentals = rentals_month(engine, month, year)
print(df_rentals)

    
    

      rental_id         rental_date  inventory_id  customer_id  \
0             1 2005-05-24 22:53:30           367          130   
1             2 2005-05-24 22:54:33          1525          459   
2             3 2005-05-24 23:03:39          1711          408   
3             4 2005-05-24 23:04:41          2452          333   
4             5 2005-05-24 23:05:21          2079          222   
...         ...                 ...           ...          ...   
1151       1153 2005-05-31 21:36:44          2725          506   
1152       1154 2005-05-31 21:42:09          2732           59   
1153       1155 2005-05-31 22:17:11          2048          251   
1154       1156 2005-05-31 22:37:34           460          106   
1155       1157 2005-05-31 22:47:45          1449           61   

             return_date  staff_id         last_update  
0    2005-05-26 22:04:30         1 2006-02-15 21:30:53  
1    2005-05-28 19:40:33         1 2006-02-15 21:30:53  
2    2005-06-01 22:12:39         1 2

In [119]:

def rental_count_month(df_rentals, month, year):
  
    df_rentals['rental_date'] = pd.to_datetime(df_rentals['rental_date'])

    df_rentals['rental_year'] = df_rentals['rental_date'].dt.year
    df_rentals['rental_month'] = df_rentals['rental_date'].dt.month

    filtered_df = df_rentals[(df_rentals['rental_year'] == year) & (df_rentals['rental_month'] == month)]
    
    rental_count = filtered_df.groupby(['rental_year', 'rental_month']).size().reset_index(name=f'rental_count_{month}_{year}')
    filtered_df = pd.merge(filtered_df, rental_count, on=['rental_year', 'rental_month'], how='left')
    
    return filtered_df



In [121]:
#Test the function
engine = mysql.connector.connect(user="root", password=password, host="localhost", database="sakila")
month = "May"
year = "2005"

df_1 =  rental_count_month(df_rentals,month,year)



#Test the function
engine = mysql.connector.connect(user="root", password=password, host="localhost", database="sakila")
month = "June"
year = "2005"

rental_count_month(df_rentals,month,year)

df_2 = rental_count_month(df_rentals,month,year)


In [125]:
def compare_rentals(df_1,df_2):
    df_compared = pd.merge(df_1, df_2, on='customer_id', how='inner')
    df_compared['difference'] = df.iloc[:, -1] - df.iloc[:, -2]
    return df_compared
    